# Visualize SAM3 + DAP3 Test Output

This notebook visualizes processed detection frames from one video output directory.

It currently includes:
- RGB frame with segmentation overlay
- Depth map with colorbar
- Paginated rendering (subset of frames at a time)

Update the config cell first (especially `VIDEO_PATH`) before running.


In [ ]:
import sys
import math
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src" / "depth_anything_3").is_dir():
            return p
    raise RuntimeError("Could not find repo root containing src/depth_anything_3")

REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_DIR:", SRC_DIR)

from depth_anything_3.utils.camera_trap_viz import (
    compute_depth_comparison_results,
    load_artifacts,
    open_video_capture,
    plot_depth_comparison,
    print_depth_statistics,
    render_page,
    select_detected_frames,
)

In [ ]:
%cd /home/dl18206/projs/Unmarked-Anything

In [ ]:
# Required inputs
OUTPUT_ROOT = Path('hpc/runs/dap3_predict_array-<run_name>')
VIDEO_STEM = None  # Set to a specific video stem string or None to auto-select first successful video from run_manifest.json
VIDEO_PATH = Path('/absolute/or/local/path/to/source_video.MP4')

# Display settings
PAGE_SIZE = 1
OVERLAY_ALPHA = 0.45
DEPTH_CMAP = 'inferno'
SHOW_BBOX_AND_CENTER = True

# Manual pagination fallback
MANUAL_PAGE = 0

# Load data and prepare pagination
video_json, npz_data = load_artifacts(OUTPUT_ROOT, VIDEO_STEM)
processed_records = select_detected_frames(video_json)
NUM_PAGES = math.ceil(len(processed_records) / PAGE_SIZE)
print(f'Pages: {NUM_PAGES} (PAGE_SIZE={PAGE_SIZE})')


In [ ]:
cap = open_video_capture(VIDEO_PATH)

In [ ]:
# Helper functions are imported from depth_anything_3.utils.camera_trap_viz

# render_page(
#     processed_records,
#     cap_obj=cap,
#     npz_obj=npz_data,
#     page=0,
#     page_size=PAGE_SIZE,
#     overlay_alpha=OVERLAY_ALPHA,
#     depth_cmap=DEPTH_CMAP,
#     show_bbox_and_center=SHOW_BBOX_AND_CENTER,
# )

In [ ]:
# Interactive processed-frame pagination with click buttons (requires ipywidgets)
try:
    import ipywidgets as widgets
    from IPython.display import display

    old_ui = globals().get('_processed_ui')
    if old_ui is not None:
        try:
            old_ui.close()
        except Exception:
            pass

    max_page = max(0, math.ceil(len(processed_records) / PAGE_SIZE) - 1)
    state = {'page': 0}
    btn_prev = widgets.Button(description='Prev Page', icon='arrow-left')
    btn_next = widgets.Button(description='Next Page', icon='arrow-right')
    status = widgets.HTML()
    out = widgets.Output()

    def _draw_page():
        page = state['page']
        status.value = f'<b>Page {page + 1}/{max_page + 1}</b>'
        btn_prev.disabled = page <= 0
        btn_next.disabled = page >= max_page
        with out:
            out.clear_output(wait=True)
            render_page(
                processed_records,
                cap_obj=cap,
                npz_obj=npz_data,
                page=page,
                page_size=PAGE_SIZE,
                overlay_alpha=OVERLAY_ALPHA,
                depth_cmap=DEPTH_CMAP,
                show_bbox_and_center=SHOW_BBOX_AND_CENTER,
            )

    def _on_prev(_):
        if state['page'] > 0:
            state['page'] -= 1
            _draw_page()

    def _on_next(_):
        if state['page'] < max_page:
            state['page'] += 1
            _draw_page()

    btn_prev.on_click(_on_prev)
    btn_next.on_click(_on_next)
    controls = widgets.HBox([btn_prev, btn_next, status])
    ui = widgets.VBox([controls, out])
    _processed_ui = ui
    display(ui)
    _draw_page()
except Exception as e:
    print('ipywidgets unavailable. Use manual pagination instead:')
    print('  render_page(processed_records, cap_obj=cap, npz_obj=npz_data, page=<k>, page_size=PAGE_SIZE)')
    print(f'Widget reason: {e}')


In [ ]:
# DAP3-style comparative analysis over all processed detection frames
analysis_records = processed_records
print(f'Analysis frames available: {len(analysis_records)}')

def render_analysis_page(idx: int) -> None:
    if idx < 0 or idx >= len(analysis_records):
        raise IndexError(f'Analysis index out of range: {idx}')
    r = compute_depth_comparison_results(
        analysis_records=[analysis_records[idx]],
        cap=cap,
        npz_data=npz_data,
    )[0]
    print_depth_statistics([r])
    plot_depth_comparison([r], depth_cmap=DEPTH_CMAP)

# Optional interactive analysis pagination with click buttons
try:
    import ipywidgets as widgets
    from IPython.display import display

    old_ui = globals().get('_analysis_ui')
    if old_ui is not None:
        try:
            old_ui.close()
        except Exception:
            pass

    max_idx = max(0, len(analysis_records) - 1)
    state = {'idx': 0}
    btn_prev = widgets.Button(description='Prev Analysis', icon='arrow-left')
    btn_next = widgets.Button(description='Next Analysis', icon='arrow-right')
    status = widgets.HTML()
    out = widgets.Output()

    def _draw_analysis():
        idx = state['idx']
        frame_idx = int(analysis_records[idx].get('frame_index', -1))
        status.value = f'<b>Analysis {idx + 1}/{max_idx + 1}</b> | frame={frame_idx}'
        btn_prev.disabled = idx <= 0
        btn_next.disabled = idx >= max_idx
        with out:
            out.clear_output(wait=True)
            render_analysis_page(idx)

    def _on_prev(_):
        if state['idx'] > 0:
            state['idx'] -= 1
            _draw_analysis()

    def _on_next(_):
        if state['idx'] < max_idx:
            state['idx'] += 1
            _draw_analysis()

    btn_prev.on_click(_on_prev)
    btn_next.on_click(_on_next)
    controls = widgets.HBox([btn_prev, btn_next, status])
    ui = widgets.VBox([controls, out])
    _analysis_ui = ui
    display(ui)
    _draw_analysis()
except Exception as e:
    print('ipywidgets unavailable. Use manual analysis pagination instead:')
    print('  render_analysis_page(<k>)  # k in [0, len(analysis_records)-1]')
    print(f'Widget reason: {e}')


In [ ]:
# Cleanup
def close_video() -> None:
    try:
        cap.release()
        print('Video handle released.')
    except NameError:
        print('No active video handle.')

close_video()
